In [3]:
!pip install mediapipe opencv-python-headless
!wget -O pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [10]:
import os
for f in os.listdir("/content"):
    print(f)

.config
pose_landmarker.task
sample_data


In [14]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import numpy as np
import json

# ============================================================
# UTILITIES
# ============================================================

def calculate_angle(a, b, c):
    """Calculate angle at point b given three [x,y] points."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))


def get_visible_side(landmarks):
    """Pick the side with better landmark visibility."""
    left_vis = np.mean([landmarks[i].visibility for i in [11, 13, 15, 23]])
    right_vis = np.mean([landmarks[i].visibility for i in [12, 14, 16, 24]])
    if left_vis >= right_vis:
        return {"side": "Left", "shoulder": 11, "elbow": 13, "wrist": 15, "hip": 23}
    else:
        return {"side": "Right", "shoulder": 12, "elbow": 14, "wrist": 16, "hip": 24}


def get_joint_coords(landmarks, idx):
    """Return [x, y] for a landmark index."""
    return [landmarks[idx].x, landmarks[idx].y]


# ============================================================
# REP DETECTOR
# ============================================================

class RepDetector:
    """Detects reps by tracking elbow angle cycles."""

    def __init__(self, extension_thresh=150, flexion_thresh=100):
        self.extension_thresh = extension_thresh  # "bottom" of pushdown
        self.flexion_thresh = flexion_thresh       # "top" of pushdown
        self.state = "idle"  # idle -> extending -> flexing -> extending ...
        self.rep_count = 0
        self.current_rep_frames = []
        self.completed_reps = []

    def update(self, elbow_angle, frame_data):
        """Feed each frame's elbow angle. Returns rep number if a rep just completed."""
        self.current_rep_frames.append(frame_data)
        completed_rep = None

        if self.state == "idle":
            if elbow_angle > self.extension_thresh:
                self.state = "extending"

        elif self.state == "extending":
            if elbow_angle < self.flexion_thresh:
                # They've bent back up — that's one full rep
                self.state = "flexing"
                self.rep_count += 1
                completed_rep = {
                    'rep_number': self.rep_count,
                    'frames': self.current_rep_frames.copy()
                }
                self.completed_reps.append(completed_rep)
                self.current_rep_frames = []

        elif self.state == "flexing":
            if elbow_angle > self.extension_thresh:
                self.state = "extending"

        return completed_rep


# ============================================================
# TRICEP PUSHDOWN SCORER
# ============================================================

class TricepPushdownScorer:
    """Scores a single rep of tricep pushdowns out of 100."""

    # --- Weights ---
    WEIGHT_ELBOW_STABILITY = 40
    WEIGHT_ROM = 30
    WEIGHT_TEMPO = 20
    WEIGHT_WRIST = 10

    @staticmethod
    def score_rep(rep_frames, fps):
        """
        Score a completed rep from its frame data.
        Each frame in rep_frames is a dict with angles and positions.
        Returns dict with total score, sub-scores, and feedback.
        """
        if len(rep_frames) < 3:
            return {'total': 0, 'feedback': ['Rep too short to analyze']}

        elbow_angles = [f['elbow_angle'] for f in rep_frames]
        upper_arm_angles = [f['upper_arm_angle'] for f in rep_frames]
        wrist_offsets = [f['wrist_offset'] for f in rep_frames]

        # ---- 1. ELBOW STABILITY (40 pts) ----
        # Measures how much the upper arm drifts during the rep
        ua_range = max(upper_arm_angles) - min(upper_arm_angles)
        ua_max = max(upper_arm_angles)

        elbow_score = 40
        elbow_feedback = None

        if ua_max > 60:
            elbow_score = 10
            elbow_feedback = "Elbows flaring significantly — pin upper arms to sides"
        elif ua_max > 45:
            elbow_score = 20
            elbow_feedback = "Elbows drifting forward — focus on keeping them stationary"
        elif ua_range > 25:
            elbow_score = 25
            elbow_feedback = "Upper arms swinging — may be using momentum"
        elif ua_range > 15:
            elbow_score = 32
            elbow_feedback = "Slight upper arm movement — minor improvement possible"
        else:
            elbow_feedback = "Upper arms stable — great control"

        # ---- 2. RANGE OF MOTION (30 pts) ----
        min_angle = min(elbow_angles)  # most bent (top of rep)
        max_angle = max(elbow_angles)  # most extended (bottom of rep)
        rom = max_angle - min_angle

        rom_score = 30
        rom_feedback = None

        if max_angle < 140:
            rom_score = 10
            rom_feedback = f"Not reaching full extension ({max_angle:.0f}°) — push all the way down"
        elif max_angle < 155:
            rom_score = 20
            rom_feedback = f"Almost full extension ({max_angle:.0f}°) — push a bit further"
        elif rom < 40:
            rom_score = 15
            rom_feedback = f"Partial rep — only {rom:.0f}° of movement"
        else:
            rom_feedback = f"Full range of motion ({rom:.0f}°) — solid"

        # ---- 3. TEMPO / CONTROL (20 pts) ----
        # Calculate frame-to-frame angle velocity
        angle_velocities = [abs(elbow_angles[i+1] - elbow_angles[i])
                           for i in range(len(elbow_angles)-1)]

        tempo_score = 20
        tempo_feedback = None

        if len(angle_velocities) > 0:
            max_velocity = max(angle_velocities)
            avg_velocity = np.mean(angle_velocities)

            # If there's a big spike relative to average, they jerked it
            if max_velocity > avg_velocity * 3.5 and max_velocity > 8:
                tempo_score = 8
                tempo_feedback = "Jerky movement detected — control the weight"
            elif max_velocity > avg_velocity * 2.5 and max_velocity > 6:
                tempo_score = 14
                tempo_feedback = "Slightly rushed — slow down for better control"
            else:
                tempo_feedback = "Smooth controlled tempo"
        else:
            tempo_feedback = "Not enough frames to judge tempo"

        # ---- 4. WRIST NEUTRALITY (10 pts) ----
        avg_wrist_offset = np.mean(wrist_offsets)
        max_wrist_offset = max(wrist_offsets)

        wrist_score = 10
        wrist_feedback = None

        if max_wrist_offset > 80:
            wrist_score = 3
            wrist_feedback = "Wrist curling significantly — keep wrists neutral"
        elif max_wrist_offset > 50:
            wrist_score = 6
            wrist_feedback = "Slight wrist deviation — try to keep straight"
        else:
            wrist_feedback = "Wrists neutral — good"

        # ---- TOTAL ----
        total = elbow_score + rom_score + tempo_score + wrist_score

        feedback_list = [elbow_feedback, rom_feedback, tempo_feedback, wrist_feedback]
        feedback_list = [f for f in feedback_list if f]

        # Identify the #1 thing to fix (lowest sub-score by percentage)
        sub_scores = {
            'elbow_stability': (elbow_score, TricepPushdownScorer.WEIGHT_ELBOW_STABILITY),
            'range_of_motion': (rom_score, TricepPushdownScorer.WEIGHT_ROM),
            'tempo_control': (tempo_score, TricepPushdownScorer.WEIGHT_TEMPO),
            'wrist_neutral': (wrist_score, TricepPushdownScorer.WEIGHT_WRIST),
        }

        worst = min(sub_scores.items(), key=lambda x: x[1][0] / x[1][1])

        return {
            'total': total,
            'elbow_stability': elbow_score,
            'range_of_motion': rom_score,
            'tempo_control': tempo_score,
            'wrist_neutral': wrist_score,
            'feedback': feedback_list,
            'biggest_issue': worst[0].replace('_', ' ').title() if worst[1][0] < worst[1][1] else None,
            'min_elbow_angle': min_angle,
            'max_elbow_angle': max_angle,
        }


# ============================================================
# SESSION SUMMARY
# ============================================================

def generate_session_summary(rep_scores):
    """Generate overall session analysis from all rep scores."""
    if not rep_scores:
        return "No reps detected."

    scores = [r['total'] for r in rep_scores]
    avg_score = np.mean(scores)
    best_rep = np.argmax(scores) + 1
    worst_rep = np.argmin(scores) + 1

    # Fatigue analysis
    if len(scores) >= 4:
        first_half = np.mean(scores[:len(scores)//2])
        second_half = np.mean(scores[len(scores)//2:])
        fatigue_drop = first_half - second_half
    else:
        fatigue_drop = 0

    # Find most common issue
    issue_counts = {}
    for r in rep_scores:
        if r['biggest_issue']:
            issue_counts[r['biggest_issue']] = issue_counts.get(r['biggest_issue'], 0) + 1

    top_issue = max(issue_counts, key=issue_counts.get) if issue_counts else None

    summary = {
        'overall_score': round(avg_score, 1),
        'total_reps': len(rep_scores),
        'best_rep': {'number': best_rep, 'score': scores[best_rep-1]},
        'worst_rep': {'number': worst_rep, 'score': scores[worst_rep-1]},
        'fatigue_drop': round(fatigue_drop, 1),
        'top_issue': top_issue,
        'rep_scores': [{'rep': i+1, 'score': s} for i, s in enumerate(scores)],
    }
    return summary


# ============================================================
# MAIN — PROCESS VIDEO
# ============================================================

base_options = python.BaseOptions(model_asset_path='pose_landmarker.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

VIDEO_PATH = "/tricep2.mp4"
OUTPUT_PATH = "/content/output_analyzed.mp4"

rep_detector = RepDetector(extension_thresh=150, flexion_thresh=100)
scorer = TricepPushdownScorer()
all_rep_scores = []

with vision.PoseLandmarker.create_from_options(options) as landmarker:
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video: {w}x{h} @ {fps:.1f} FPS, {total} frames\n")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (w, h))

    frame_index = 0
    current_form_display = {"overall": "", "feedback": [], "color": (255,255,255)}

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = int(frame_index * 1000 / fps)

        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.pose_landmarks:
            lm = result.pose_landmarks[0]
            side = get_visible_side(lm)

            shoulder = get_joint_coords(lm, side['shoulder'])
            elbow = get_joint_coords(lm, side['elbow'])
            wrist = get_joint_coords(lm, side['wrist'])
            hip = get_joint_coords(lm, side['hip'])

            elbow_angle = calculate_angle(shoulder, elbow, wrist)
            upper_arm_angle = calculate_angle(hip, shoulder, elbow)
            wrist_offset = abs(wrist[0] - elbow[0]) * w

            # Build frame data for the rep detector
            frame_data = {
                'frame': frame_index,
                'elbow_angle': elbow_angle,
                'upper_arm_angle': upper_arm_angle,
                'wrist_offset': wrist_offset,
            }

            # Check if a rep just completed
            completed = rep_detector.update(elbow_angle, frame_data)

            if completed:
                rep_score = scorer.score_rep(completed['frames'], fps)
                rep_score['rep_number'] = completed['rep_number']
                all_rep_scores.append(rep_score)

                # Update display info
                score = rep_score['total']
                if score >= 80:
                    color = (0, 255, 0)
                elif score >= 60:
                    color = (0, 200, 255)
                else:
                    color = (0, 0, 255)

                current_form_display = {
                    "overall": f"Rep {completed['rep_number']}: {score}/100",
                    "feedback": rep_score['feedback'][:2],  # show top 2 feedback items
                    "color": color,
                }

                print(f"Rep {completed['rep_number']}: {score}/100")
                for fb in rep_score['feedback']:
                    print(f"   {fb}")
                print()

            # ---- DRAW ON FRAME ----
            joints = {
                'hip': (int(hip[0]*w), int(hip[1]*h)),
                'shoulder': (int(shoulder[0]*w), int(shoulder[1]*h)),
                'elbow': (int(elbow[0]*w), int(elbow[1]*h)),
                'wrist': (int(wrist[0]*w), int(wrist[1]*h)),
            }

            # Skeleton
            cv2.line(frame, joints['hip'], joints['shoulder'], (255, 255, 0), 2)
            cv2.line(frame, joints['shoulder'], joints['elbow'], (255, 255, 0), 2)
            cv2.line(frame, joints['elbow'], joints['wrist'], (255, 255, 0), 2)

            for pt in joints.values():
                cv2.circle(frame, pt, 6, current_form_display['color'], -1)

            # Angle readouts
            cv2.putText(frame, f"{elbow_angle:.0f} deg",
                        (joints['elbow'][0]+10, joints['elbow'][1]-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

            # Rep count
            cv2.putText(frame, f"Reps: {rep_detector.rep_count}",
                        (w - 180, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

            # Form verdict (persists until next rep)
            if current_form_display['overall']:
                cv2.putText(frame, current_form_display['overall'], (30, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, current_form_display['color'], 3)
                for i, fb in enumerate(current_form_display['feedback']):
                    cv2.putText(frame, fb, (30, 85 + i*28),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 2)

        out.write(frame)
        frame_index += 1

    cap.release()
    out.release()

# ============================================================
# SESSION SUMMARY
# ============================================================

print("=" * 50)
print("SESSION SUMMARY")
print("=" * 50)

if all_rep_scores:
    summary = generate_session_summary(all_rep_scores)

    print(f"\nOverall Score: {summary['overall_score']}/100")
    print(f"Total Reps: {summary['total_reps']}")
    print(f"Best Rep: #{summary['best_rep']['number']} ({summary['best_rep']['score']}/100)")
    print(f"Worst Rep: #{summary['worst_rep']['number']} ({summary['worst_rep']['score']}/100)")

    if summary['fatigue_drop'] > 5:
        print(f"\nFatigue detected: Form dropped ~{summary['fatigue_drop']} points in second half of set")
    elif summary['fatigue_drop'] > 0:
        print(f"\nMinimal fatigue: Consistent form throughout the set")

    if summary['top_issue']:
        print(f"\n#1 Thing to work on: {summary['top_issue']}")

    print("\nRep-by-Rep:")
    for r in all_rep_scores:
        print(f"  Rep {r['rep_number']}: {r['total']}/100 — {r['feedback'][0]}")

    print(f"\nAnnotated video saved to: {OUTPUT_PATH}")
else:
    print("No reps detected. Check that the video shows a clear side view of the exercise.")

Video: 640x360 @ 25.0 FPS, 397 frames

Rep 1: 84/100
   Upper arms stable — great control
   Full range of motion (104°) — solid
   Jerky movement detected — control the weight
   Slight wrist deviation — try to keep straight

Rep 2: 84/100
   Upper arms stable — great control
   Full range of motion (108°) — solid
   Jerky movement detected — control the weight
   Slight wrist deviation — try to keep straight

Rep 3: 66/100
   Upper arms swinging — may be using momentum
   Full range of motion (169°) — solid
   Jerky movement detected — control the weight
   Wrist curling significantly — keep wrists neutral

Rep 4: 81/100
   Upper arms stable — great control
   Full range of motion (119°) — solid
   Jerky movement detected — control the weight
   Wrist curling significantly — keep wrists neutral

SESSION SUMMARY

Overall Score: 78.8/100
Total Reps: 4
Best Rep: #1 (84/100)
Worst Rep: #3 (66/100)

Fatigue detected: Form dropped ~10.5 points in second half of set

#1 Thing to work on: Te

In [13]:
# Export session data as JSON (for future app integration)
if all_rep_scores:
    export_data = {
        'exercise': 'Tricep Pushdown',
        'summary': summary,
        'reps': all_rep_scores,
    }

    with open('/content/session_report.json', 'w') as f:
        json.dump(export_data, f, indent=2, default=str)

    print("Session report saved to /content/session_report.json")
    print("\nJSON preview:")
    print(json.dumps(export_data, indent=2, default=str)[:1500])

Session report saved to /content/session_report.json

JSON preview:
{
  "exercise": "Tricep Pushdown",
  "summary": {
    "overall_score": 76.0,
    "total_reps": 3,
    "best_rep": {
      "number": "1",
      "score": 76
    },
    "worst_rep": {
      "number": "1",
      "score": 76
    },
    "fatigue_drop": 0,
    "top_issue": "Tempo Control",
    "rep_scores": [
      {
        "rep": 1,
        "score": 76
      },
      {
        "rep": 2,
        "score": 76
      },
      {
        "rep": 3,
        "score": 76
      }
    ]
  },
  "reps": [
    {
      "total": 76,
      "elbow_stability": 32,
      "range_of_motion": 30,
      "tempo_control": 8,
      "wrist_neutral": 6,
      "feedback": [
        "Slight upper arm movement \u2014 minor improvement possible",
        "Full range of motion (91\u00b0) \u2014 solid",
        "Jerky movement detected \u2014 control the weight",
        "Slight wrist deviation \u2014 try to keep straight"
      ],
      "biggest_issue": "Temp